In [6]:
import requests
import pandas as pd
import time
import os

In [2]:
app_ids = [
    730, 570, 440, 578080, 271590,
    1172470, 1085660, 381210, 1091500, 1245620,
    1938090, 252490, 1599340, 413150, 814380,
    292030, 553850, 236390, 394360, 1086940
]

In [3]:
def get_current_players(app_id):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        return {
            "app_id": app_id,
            "current_players": data.get("response", {}).get("player_count")
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return {
            "app_id": app_id,
            "current_players": None
        }

In [4]:
engagement_records = []

for app_id in app_ids:
    row = get_current_players(app_id)
    engagement_records.append(row)
    time.sleep(1)

df_engagement = pd.DataFrame(engagement_records)
df_engagement.head()

,app_id,current_players
0,730,1380146
1,570,540425
2,440,46100
3,578080,280946
4,271590,79496


In [5]:
print(df_engagement.shape)
print(df_engagement.info())
print(df_engagement.isnull().sum())
print(df_engagement)

(20, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   app_id           20 non-null     int64
 1   current_players  20 non-null     int64
dtypes: int64(2)
memory usage: 452.0 bytes
None
app_id             0
current_players    0
dtype: int64
     app_id  current_players
0       730          1380146
1       570           540425
2       440            46100
3    578080           280946
4    271590            79496
5   1172470            90012
6   1085660             8430
7    381210            53570
8   1091500            28670
9   1245620            31339
10  1938090            34347
11   252490           115510
12  1599340            12536
13   413150            49496
14   814380             2666
15   292030            14761
16   553850            37167
17   236390            64659
18   394360            41854
19  1086940            45559


In [7]:
os.makedirs("../data/raw", exist_ok=True)
df_engagement.to_csv("../data/raw/steam_engagement_batch1.csv", index=False)

print("Saved raw engagement batch.")

Saved raw engagement batch.


In [8]:
df_engagement_clean = df_engagement.copy()

df_engagement_clean.columns = df_engagement_clean.columns.str.strip().str.lower()
df_engagement_clean = df_engagement_clean.drop_duplicates(subset="app_id")
df_engagement_clean["current_players"] = pd.to_numeric(df_engagement_clean["current_players"], errors="coerce")

os.makedirs("../data/cleaned", exist_ok=True)
df_engagement_clean.to_csv("../data/cleaned/steam_engagement_clean.csv", index=False)

print("Saved cleaned engagement data.")

Saved cleaned engagement data.
